In [ ]:
import os
import torch
from dotenv import load_dotenv

load_dotenv()
access_token = os.environ.get("HF_TOKEN")
if access_token is None:
    raise ValueError("Set HF_TOKEN in your environment or in a local .env file.")

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

config = {
    "tokenizer": tokenizer,
    "batch_size": 4,
    "context_length": 512,
    "vocab_size": len(tokenizer),
    "embedding_dim": 768,
    "n_layers": 8, 
    "n_heads": 12,
    "n_kv_heads": 6,
}

batch_size = config["batch_size"]
context_length = config["context_length"]
vocab_size = config["vocab_size"]
embedding_dim = config["embedding_dim"]
n_layers = config["n_layers"]
n_heads = config["n_heads"]
n_kv_heads = config["n_kv_heads"]

In [ ]:
from data import create_python_dataset, CoderDataset
from torch.utils.data import DataLoader

small_dataset = create_python_dataset("data/python-edu/train.jsonl").take(20)
coder_dataset = CoderDataset(
    dataset=small_dataset,
    tokenizer=tokenizer,
    context_length=context_length
)

data_loader = DataLoader(
    dataset=coder_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

x_batch, y_batch = next(iter(data_loader))
print(f"x batch shape is:", x_batch.shape)
print(f"y batch shape is:", y_batch.shape)

In [ ]:
token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
tok_embeddings = token_embedding_layer(x_batch)
tok_embeddings.shape

In [ ]:
pos_embedding_layer = torch.nn.Embedding(context_length, embedding_dim)
pos_embeddings = pos_embedding_layer(torch.arange(x_batch.shape[1]))
pos_embeddings.shape

In [ ]:
x = tok_embeddings + pos_embeddings
x.shape

In [ ]:
from torch import nn

class Attention(nn.Module):
    def __init__(self, embedding_dim, n_heads, n_kv_heads, dropout=0.0, is_causal=True):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_rep = n_heads // n_kv_heads
        assert embedding_dim % n_heads == 0, "embedding_dim must be divisible by n_heads"
        self.head_dim = embedding_dim // n_heads
        self.dropout = dropout
        self.is_causal = is_causal

        self.query_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)
        self.key_proj = nn.Linear(embedding_dim, n_kv_heads * self.head_dim, bias=False)
        self.value_proj = nn.Linear(embedding_dim, n_kv_heads * self.head_dim, bias=False)

        self.output_proj = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, x):
        # We take input with shape (batch_size, context_length, embedding_dim)..
        # We want to project query, key, and value matrices from the input
        # We then reshape and transpose them to have shape (batch_size, n_head, context_length, head_dim)
        # We calculate attention with each of those heads
        # Then we merge the heads back to shape (batch_size, context_length, embedding_dim) and return that

        batch_size, context_length, embedding_dim = x.shape
        query_matrix = self.query_proj(x)
        key_matrix = self.key_proj(x)
        value_matrix = self.value_proj(x)

        query_heads = query_matrix.reshape(batch_size, context_length, self.n_heads, self.head_dim).transpose(1, 2)
        key_heads = key_matrix.reshape(batch_size, context_length, self.n_kv_heads, self.head_dim).transpose(1, 2)
        value_heads = value_matrix.reshape(batch_size, context_length, self.n_kv_heads, self.head_dim).transpose(1, 2)

        key_heads = key_heads.repeat_interleave(self.n_rep, dim=1)
        value_heads = value_heads.repeat_interleave(self.n_rep, dim=1)

        grouped_query_attention = nn.functional.scaled_dot_product_attention(
            query_heads,
            key_heads,
            value_heads,
            dropout_p = self.dropout if self.training else 0.0,
            is_causal=self.is_causal
        )

        merged_attention = grouped_query_attention.transpose(1, 2).flatten(start_dim=2)
        return self.output_proj(merged_attention)



attention = Attention(embedding_dim, n_heads, n_kv_heads, 0.)
attention_output = attention(x)
attention_output.shape


In [ ]:
from torch import nn
import torch.nn.functional as F

class FFN(nn.Module):
    def __init__(self, embedding_dim, intermediate_dim):
        super().__init__()

        self.gate_proj = nn.Linear(embedding_dim, intermediate_dim, bias=False)
        self.up_proj = nn.Linear(embedding_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, embedding_dim, bias=False)

    def forward(self, x):
        
        gated_output = self.gate_proj(x)
        up_output = self.up_proj(x)

        return self.down_proj(F.silu(gated_output) * up_output)


feed_forward = FFN(embedding_dim, 2048)
ffn_output = feed_forward(attention_output)
ffn_output.shape

In [ ]:
from torch import nn

class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, n_heads, n_kv_heads, intermediate_dim, dropout=0.0, is_causal=True):
        super().__init__()
        
        self.norm_1 = nn.RMSNorm(embedding_dim, eps=1e-6)
        self.attention = Attention(embedding_dim, n_heads, n_kv_heads, dropout, is_causal)
        self.norm_2 = nn.RMSNorm(embedding_dim, eps=1e-6)
        self.ffn = FFN(embedding_dim, intermediate_dim)
    
    def forward(self, x):
        res_x_1 = x
        x = self.norm_1(x)
        x = self.attention(x)
        x = x + res_x_1
        res_x_2 = x
        x = self.norm_2(x)
        x = self.ffn(x)
        return res_x_2 + x


one_transfomer = TransformerBlock(embedding_dim, n_heads, n_kv_heads, 2048)
output_trans = one_transfomer(x)
output_trans.shape

In [ ]:
from torch import nn

class LLM(nn.Module):
    def __init__(self, vocab_size, context_length, embedding_dim, n_layers, n_heads, n_kv_heads, intermediate_dim, dropout = 0.0, is_causal=True):
        super().__init__()
        self.context_length = context_length

        self.tok_embedding = nn.Embedding(vocab_size, embedding_dim) 
        self.pos_embedding = nn.Embedding(context_length, embedding_dim) 
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, n_heads, n_kv_heads, intermediate_dim, dropout, is_causal)
            for _ in range(n_layers)
        ])
        
        self.final_norm = nn.RMSNorm(embedding_dim, eps=1e-6)

        self.output_proj = nn.Linear(embedding_dim, vocab_size, bias=False)
        self.output_proj.weight = self.tok_embedding.weight
    
    def forward(self, x):
        batch_size, context_length = x.shape
        pos_embeddings = self.pos_embedding(torch.arange(context_length, device=x.device))
        x = self.tok_embedding(x) + pos_embeddings
        
        for block in self.blocks:
            x = block(x)
        
        x = self.final_norm(x)
        return self.output_proj(x)
    
coder_llm = LLM(vocab_size, context_length, embedding_dim, n_layers, n_heads, n_kv_heads, 2048)
logits = coder_llm(x_batch)
logits.shape
output = torch.argmax(F.softmax(logits, dim=-1), dim=-1)
print(logits.shape)
print(output.shape)


        